# การสังเคราะห์ข้อมูลตัวอย่างกลุ่มน้อยด้วย Borderline-SMOTE (CENTRAL)

Notebook นี้สาธิตและอธิบายขั้นตอนการใช้ **Borderline-SMOTE** 
เพื่อเพิ่มประสิทธิภาพการตรวจจับคลาสที่มีตัวอย่างน้อย (Minority Classes) โดยเฉพาะ **Class 4: Credential Abuse (R2L)**
ให้มีตัวอย่างเพิ่มขึ้นเป็น **40,000 โฟลว์** บนพื้นที่ฟีเจอร์ 19 มิติ โดยไม่สร้าง Noise ในพื้นที่ของ Benign Traffic


## 1. หลักการทางทฤษฎี: SMOTE ธรรมดา vs Borderline-SMOTE

- **Standard SMOTE**: สุ่มจับคู่ตัวอย่างในคลาสเดียวกันทั้งหมด ทำให้เกิดจุดสังเคราะห์รุกล้ำเข้าไปในคลัสเตอร์ของคลาสส่วนใหญ่ (Benign) ส่งผลให้ False Positive Rate สูงขึ้น
- **Borderline-SMOTE**: คัดกรองเฉพาะตัวอย่างที่อยู่บริเวณ **ขอบเขตการตัดสินใจ (Decision Boundary / DANGER Zone)** โดยพิจารณาจากจำนวนเพื่อนบ้าน $m$ ตัว หากมีเพื่อนบ้านต่างคลาสอยู่ระหว่าง $k/2 \le m \le k$ จุดนั้นจะถูกจัดเป็นขอบเขต และจะทำการสร้างตัวอย่างสังเคราะห์เฉพาะบริเวณนี้


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

smote_dir = Path("output/smote")
with open(smote_dir / "smote_report.json") as fp:
    report = json.load(fp)

print("Borderline-SMOTE Execution Report:")
for k, v in report.items():
    print(f"  {k}: {v}")


## 2. ตรวจสอบการกระจายตัวของคลาสก่อนและหลังการสุ่มตัวอย่าง


In [ ]:
df_before = pd.Series(report["class_distribution_before"], name="Before SMOTE")
df_after = pd.Series(report["class_distribution_after"], name="After SMOTE")

comparison = pd.concat([df_before, df_after], axis=1)
comparison["Net Increase"] = comparison["After SMOTE"] - comparison["Before SMOTE"]
display(comparison)


## 3. ตรวจสอบความถูกต้องของข้อมูลสังเคราะห์ (Synthetic Flows Inspection)


In [ ]:
labels_smote = pd.read_csv(smote_dir / "labels.csv", low_memory=False)
synthetic_mask = labels_smote["is_synthetic"] == True
print(f"Total Synthetic Rows Generated: {synthetic_mask.sum():,}")
print("\nSample Synthetic Labels:")
display(labels_smote[synthetic_mask].head(5))
